# 04 - Frozen Qwen3-Embedding + LightGBM (smoke test first)

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import features, models
SMOKE = True
idx = X.index[:64] if SMOKE else X.index
with evaluate.log_run("qwen3-embedding", 0, "encode", is_synthetic=SYN, smoke=SMOKE):
    emb = features.embed(X.loc[idx, "text"])
np.save(config.OUTPUTS / ("emb_smoke.npy" if SMOKE else "emb.npy"), emb)

In [ ]:
if not SMOKE:
    for seed in config.SEEDS_FINAL:
        with evaluate.log_run("emb-lgbm", seed, "----", is_synthetic=SYN):
            r = models.embeddings_lgbm(emb, y, split, seed=seed)
        evaluate.save_scores("emb-lgbm", seed, r["scores"], is_synthetic=SYN)
    evaluate.lock_from_val_a("emb-lgbm", y, {"n_estimators": 800}, is_synthetic=SYN)